# 01 - Clean Enrollment (CRDC 2017–18)
**Objective:** Ingest enrollment data, filter to operational high schools (grades 9–12 / ungraded HS), resolve federal suppression codes, and engineer demographic rates (IDEA, 504, LEP).

### Key Preprocessing Rules
1. **ID Formatting:** Pad `COMBOKEY` to 12 digits (`str.zfill(12)`) to restore dropped leading zeros.
2. **High School Filter:** Filter for high schools using grades 9–12 and `SCH_UGDETAIL_HS` from `School Characteristics.csv`.
3. **Suppression Codes:** Map negative codes (`-9`, `-5`, `-3`) to `NaN`.
4. **Denominator:** Compute `TOT_ENR = TOT_ENR_M + TOT_ENR_F` while preserving true missing values (`both_nan`).
5. **Feature Engineering:** Normalize demographic counts by `TOT_ENR` and cap at `1.0` via `.clip(upper=1.0)` to handle reporting anomalies.

In [2]:
import os
import pandas as pd
import numpy as np

def get_high_school_combokeys(filepath: str) -> set:
  """
    Scans School Characteristics.csv, preserves 12-digit COMBOKEY strings,
    and returns a set of COMBOKEYs for all high schools (Grades 9-12 and Ungraded HS).
  """
  target_cols = [
        "SCH_GRADE_G09", 
        "SCH_GRADE_G10", 
        "SCH_GRADE_G11", 
        "SCH_GRADE_G12",
        "SCH_UGDETAIL_HS"
  ]
  
  df_char = pd.read_csv(filepath, 
                        usecols=["COMBOKEY"] + target_cols,
                        dtype={"COMBOKEY": str},
                        encoding="latin1"
  )
  
  # Restores dropped leading zero from 11-digit IDs (e.g. Alabama '01')
  df_char["COMBOKEY"] = df_char["COMBOKEY"].astype(str).str.zfill(12)
  
  is_highschool = df_char[target_cols].isin(["Yes", "YES", "Y", 1, "1"]).any(axis=1)
  
  hs_keys = set(df_char.loc[is_highschool, "COMBOKEY"])
  print(f"[School Characteristics] Identified {len(hs_keys):,} High Schools.")
  return hs_keys

In [10]:
DATA_DIR = os.path.join("..", "data")
CLEANED_DIR = os.path.join("..", "cleaned")
CHAR_PATH = os.path.join(DATA_DIR, "School Characteristics.csv")
ENR_PATH = os.path.join(DATA_DIR, "Enrollment.csv")
OUTPUT_PATH = os.path.join(CLEANED_DIR, "filtered_Enrollment_1718.csv")

# Get the set of combokeys that belongs to high schools
hs_combokeys = get_high_school_combokeys(CHAR_PATH)

target_cols = [
    "COMBOKEY", "LEAID", "SCHID", "SCH_NAME", "LEA_NAME", "LEA_STATE", "JJ",
    "TOT_ENR_M", "TOT_ENR_F",
    "SCH_ENR_IDEA_M", "SCH_ENR_IDEA_F",
    "SCH_ENR_504_M", "SCH_ENR_504_F",
    "SCH_ENR_LEP_M", "SCH_ENR_LEP_F"
]

# Load the csv with the target cols
df_enr = pd.read_csv(ENR_PATH,
                       usecols=target_cols,
                       dtype={"COMBOKEY": str, "LEAID": str, "SCHID": str},
                       encoding="latin1"
)

# Repair leading zeros across all identifier columns
df_enr["COMBOKEY"] = df_enr["COMBOKEY"].astype(str).str.zfill(12)
df_enr["LEAID"] = df_enr["LEAID"].astype(str).str.zfill(7)
df_enr["SCHID"] = df_enr["SCHID"].astype(str).str.zfill(5)

# Filter by High Schools
df_enr = df_enr[df_enr["COMBOKEY"].isin(hs_combokeys)].copy()
print(f"[Enrollment] High schools: {df_enr.shape}")

# Turn negative CRDC suppression codes (-9, -5, -3) to NaN
text_cols = ["COMBOKEY", "LEAID", "SCHID", "SCH_NAME", "LEA_NAME", "LEA_STATE", "JJ"]
num_cols = [col for col in target_cols if col not in text_cols]
#print(num_cols)

for col in num_cols:
  df_enr[col] = pd.to_numeric(df_enr[col], errors="coerce")
  df_enr[col] = df_enr[col].map(lambda x: np.nan if x < 0 else x)
  
df_enr["TOT_ENR"] = df_enr["TOT_ENR_M"].fillna(0) + df_enr["TOT_ENR_F"].fillna(0)

# Filter out the non reporting schools. No data reported for both male and females. (there's 2)
both_nan = df_enr["TOT_ENR_M"].isna() & df_enr["TOT_ENR_F"].isna()
print(f"Schools missing BOTH male and female enrollment: {both_nan.sum():,}")
df_enr.loc[both_nan, "TOT_ENR"] = np.nan

# Keep schools that are operational
df_enr = df_enr[df_enr["TOT_ENR"] > 0].copy()

# Calculate subgroup totals and rates
# any value greater than 1.0 is capped down to 1.0
df_enr["TOT_ENR_IDEA"] = df_enr["SCH_ENR_IDEA_M"].fillna(0) + df_enr["SCH_ENR_IDEA_F"].fillna(0)
df_enr["IDEA_RATE"] = (df_enr["TOT_ENR_IDEA"] / df_enr["TOT_ENR"]).clip(upper=1.0)

df_enr["TOT_ENR_504"] = df_enr["SCH_ENR_504_M"].fillna(0) + df_enr["SCH_ENR_504_F"].fillna(0)
df_enr["SECTION_504_RATE"] = (df_enr["TOT_ENR_504"] / df_enr["TOT_ENR"]).clip(upper=1.0)

df_enr["TOT_ENR_LEP"] = df_enr["SCH_ENR_LEP_M"].fillna(0) + df_enr["SCH_ENR_LEP_F"].fillna(0)
df_enr["LEP_RATE"] = (df_enr["TOT_ENR_LEP"] / df_enr["TOT_ENR"]).clip(upper=1.0)

df_enr["YEAR"] = "2017-2018"

export_cols = [
    # 1. Identifiers
    "YEAR", "COMBOKEY", "LEAID", "SCHID", "SCH_NAME", "LEA_NAME", "LEA_STATE", "JJ",
    
    # 2. Total School Enrollment
    "TOT_ENR", "TOT_ENR_M", "TOT_ENR_F",
    
    # 3. IDEA Special Education
    "SCH_ENR_IDEA_M", "SCH_ENR_IDEA_F", "TOT_ENR_IDEA", "IDEA_RATE",
    
    # 4. Section 504 Disability
    "SCH_ENR_504_M", "SCH_ENR_504_F", "TOT_ENR_504", "SECTION_504_RATE",
    
    # 5. English Language Learners / LEP
    "SCH_ENR_LEP_M", "SCH_ENR_LEP_F", "TOT_ENR_LEP", "LEP_RATE"
]

df_clean = df_enr[export_cols].copy()
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved cleaned high school enrollment data to: {os.path.abspath(OUTPUT_PATH)}")
print(f"Final Shape: {df_clean.shape}")
print(df_clean.head())

[School Characteristics] Identified 26,515 High Schools.
[Enrollment] High schools: (26515, 15)
Schools missing BOTH male and female enrollment: 2

Saved cleaned high school enrollment data to: /Users/admin/Capital-One-1B-ai-driven-resource-allocation-to-reduce-restrictive-practices/cleaned/filtered_Enrollment_1718.csv
Final Shape: (26513, 23)
        YEAR      COMBOKEY    LEAID  SCHID                       SCH_NAME  \
0  2017-2018  010000201705  0100002  01705  Wallace Sch - Mt Meigs Campus   
1  2017-2018  010000201706  0100002  01706      McNeel Sch - Vacca Campus   
2  2017-2018  010000201876  0100002  01876         Alabama Youth Services   
3  2017-2018  010000299995  0100002  99995                 AUTAUGA CAMPUS   
5  2017-2018  010000500871  0100005  00871        Albertville High School   

                 LEA_NAME LEA_STATE   JJ  TOT_ENR  TOT_ENR_M  ...  \
0  Alabama Youth Services        AL  Yes    133.0      133.0  ...   
1  Alabama Youth Services        AL  Yes     58.0    